# 01. Exploratory Data Analysis (EDA) — Diabetic Retinopathy Fundus Screening

> **Marconi Research & Innovations Lab — Internship Capstone Project**  
> **Dataset:** APTOS 2019 Blindness Detection (`sngsfydy/aptos` / EyePACS)

This notebook performs comprehensive exploratory data analysis on retinal fundus photographs for diabetic retinopathy (DR) severity classification. We investigate:
1. Dataset structure and feature schema
2. Class distribution and severity imbalance
3. Representative fundus image visualization across all 5 clinical grades
4. Image dimensions and resolution variability
5. Color channel (RGB) characteristics
6. Preprocessing techniques (Black boundary cropping & Ben Graham feature enhancement)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import datasets
from collections import Counter

# Plot style settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("Libraries successfully imported!")

## 1. Dataset Loading & Structure

We load the APTOS 2019 Blindness Detection dataset via Hugging Face `datasets`.

In [ ]:
# Load APTOS dataset
dataset_name = "sngsfydy/aptos"
print(f"Loading dataset: {dataset_name}...")
dataset = datasets.load_dataset(dataset_name, split="train")

print(f"Total samples: {len(dataset):,}")
print(f"Features schema: {dataset.features}")
print(f"Sample example: {dataset[0]}")

## 2. Class Distribution & Imbalance Analysis

Diabetic Retinopathy severity is graded using the International Clinical Diabetic Retinopathy (ICDR) scale:
* **0 — No DR:** Normal healthy fundus, no microaneurysms.
* **1 — Mild:** Microaneurysms only.
* **2 — Moderate:** More than just microaneurysms, but less than severe DR (exudates, cotton wool spots).
* **3 — Severe:** >20 intraretinal hemorrhages in each of 4 quadrants, or definite venous beading in 2+ quadrants, or prominent IRMA in 1+ quadrant.
* **4 — Proliferative DR:** Neovascularization, vitreous/preretinal hemorrhage.

In [ ]:
class_names = {
    0: "0 - No DR",
    1: "1 - Mild",
    2: "2 - Moderate",
    3: "3 - Severe",
    4: "4 - Proliferative"
}

# Extract labels
labels = [dataset[i]['label'] for i in range(len(dataset))]
label_counts = Counter(labels)

df_dist = pd.DataFrame([
    {
        "Class ID": k,
        "Severity": class_names[k],
        "Count": label_counts[k],
        "Percentage (%)": round((label_counts[k] / len(labels)) * 100, 2)
    }
    for k in sorted(class_names.keys())
])

print("=== Class Distribution Table ===")
print(df_dist.to_string(index=False))

# Visualizing distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

palette = ["#2ecc71", "#3498db", "#f39c12", "#e67e22", "#e74c3c"]
bars = ax1.bar(df_dist["Severity"], df_dist["Count"], color=palette, edgecolor='black', alpha=0.85)
ax1.set_title("Diabetic Retinopathy Class Distribution (Counts)", fontweight="bold", fontsize=12)
ax1.set_ylabel("Number of Fundus Images")
ax1.set_xticklabels(df_dist["Severity"], rotation=20, ha="right")

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'{height}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords="offset points",
                 ha='center', va='bottom', fontweight='bold')

# Donut chart
wedges, texts, autotexts = ax2.pie(
    df_dist["Count"], labels=df_dist["Severity"], autopct='%1.1f%%',
    startangle=140, colors=palette, textprops={'fontsize': 9},
    wedgeprops=dict(width=0.4, edgecolor='white')
)
ax2.set_title("Proportion per DR Severity Class", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

## 3. Visualizing Fundus Images Across Severity Grades

Let us inspect actual retinal photographs corresponding to each severity stage.

In [ ]:
# Find representative sample indices for each class
sample_indices = {}
for idx in range(len(dataset)):
    lbl = dataset[idx]['label']
    if lbl not in sample_indices:
        sample_indices[lbl] = idx
    if len(sample_indices) == 5:
        break

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, (lbl, idx) in enumerate(sorted(sample_indices.items())):
    item = dataset[idx]
    img = item['image']
    axes[i].imshow(img)
    axes[i].set_title(f"{class_names[lbl]}\n(Idx: {idx})", fontsize=11, fontweight='bold', color='navy')
    axes[i].axis('off')

plt.suptitle("Representative Retinal Fundus Images by Diabetic Retinopathy Grade", fontsize=14, y=1.05, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Image Dimensions & Resolution Analysis

Fundus cameras produce varying image resolutions and aspect ratios. Understanding resolution variation is crucial for designing resizing and normalization pipelines.

In [ ]:
# Sample image resolutions
num_samples_to_check = min(300, len(dataset))
widths, heights, aspect_ratios = [], [], []

for i in range(num_samples_to_check):
    img = dataset[i]['image']
    w, h = img.size
    widths.append(w)
    heights.append(h)
    aspect_ratios.append(round(w / h, 2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.scatter(widths, heights, alpha=0.6, c='teal', edgecolors='k', s=40)
ax1.set_title("Image Dimensions (Width vs. Height)", fontweight="bold")
ax1.set_xlabel("Width (pixels)")
ax1.set_ylabel("Height (pixels)")
ax1.grid(True, linestyle="--", alpha=0.5)

sns.histplot(aspect_ratios, ax=ax2, color='crimson', kde=True, bins=15)
ax2.set_title("Aspect Ratio Distribution (Width / Height)", fontweight="bold")
ax2.set_xlabel("Aspect Ratio")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Common resolutions inspected: {Counter(zip(widths[:50], heights[:50]))}")

## 5. Color Channel (RGB) Analysis

In fundus photographs:
* The **Green channel** provides the highest contrast for blood vessels, hemorrhages, and microaneurysms.
* The **Red channel** often saturates due to high choroidal reflection.
* The **Blue channel** often has high noise and lower signal.

In [ ]:
# Compare channels of a normal fundus vs severe DR fundus
normal_img = np.array(dataset[sample_indices[0]]['image'])
severe_img = np.array(dataset[sample_indices[4]]['image'])

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Row 1: Normal (Class 0)
axes[0, 0].imshow(normal_img)
axes[0, 0].set_title("Normal Fundus (RGB)", fontweight='bold')
axes[0, 1].imshow(normal_img[:, :, 0], cmap='Reds')
axes[0, 1].set_title("Red Channel")
axes[0, 2].imshow(normal_img[:, :, 1], cmap='Greens')
axes[0, 2].set_title("Green Channel (High Contrast)")
axes[0, 3].imshow(normal_img[:, :, 2], cmap='Blues')
axes[0, 3].set_title("Blue Channel")

# Row 2: Proliferative DR (Class 4)
axes[1, 0].imshow(severe_img)
axes[1, 0].set_title("Proliferative DR (RGB)", fontweight='bold')
axes[1, 1].imshow(severe_img[:, :, 0], cmap='Reds')
axes[1, 1].set_title("Red Channel")
axes[1, 2].imshow(severe_img[:, :, 1], cmap='Greens')
axes[1, 2].set_title("Green Channel (High Contrast)")
axes[1, 3].imshow(severe_img[:, :, 2], cmap='Blues')
axes[1, 3].set_title("Blue Channel")

for ax in axes.flat:
    ax.axis('off')

plt.suptitle("Color Channel Breakdown in Normal vs Proliferative Diabetic Retinopathy", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Preprocessing & Feature Enhancement (Ben Graham Method)

Retinal fundus images often have varying illumination and black border padding.  
Ben Graham's preprocessing technique:
1. Crops uninformative black background.
2. Subtracts local Gaussian blurred color to normalize illumination and accentuate lesions.

In [ ]:
from src.preprocessing import crop_image_from_gray, ben_graham_preprocessing

# Test preprocessing on a sample image
sample_pil = dataset[sample_indices[2]]['image']
sample_np = np.array(sample_pil)

# 1. Original
orig = sample_pil.resize((224, 224))

# 2. Cropped
cropped_np = crop_image_from_gray(sample_np)
cropped_pil = Image.fromarray(cropped_np).resize((224, 224))

# 3. Ben Graham Enhanced
ben_graham_pil = ben_graham_preprocessing(sample_pil, img_size=224, sigma_x=10)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(orig)
axes[0].set_title("1. Original Resized (224x224)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cropped_pil)
axes[1].set_title("2. Black Border Cropped", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(ben_graham_pil)
axes[2].set_title("3. Ben Graham Color Subtracted", fontweight='bold')
axes[2].axis('off')

plt.suptitle("Preprocessing Pipeline Comparison for Retinal Fundus Photographs", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Key Findings & Clinical Modeling Strategy

| Observation | Impact on ML Pipeline | Engineering Solution |
| :--- | :--- | :--- |
| **Significant Class Imbalance** | Class 0 (No DR) represents ~49% of the dataset, while Class 3 (Severe) and Class 1 (Mild) are minority classes. | Use **Macro-averaged F1**, **Sensitivity**, and **Class-weighted Cross-Entropy loss** rather than pure accuracy. |
| **Illumination Variance** | Varying lighting across fundus cameras can cause domain shifts. | Apply **Ben Graham local color subtraction** or **ColorJitter data augmentation**. |
| **High-frequency Lesions** | Microaneurysms and exudates are small, localized features. | Use modern multi-scale architectures (**ConvNeXt**, **DenseNet**, **Swin Transformer**) with skip connections and feature reuse. |
| **Deployment Constraints** | Screening requires low latency in clinics. | Benchmark **Inference Latency (ms)** and **Model Size (MB)** alongside clinical sensitivity. |